# 🔄 Running Policies in Simulation

This is where everything comes together: a **policy** controlling a **robot**
inside a **physics simulation**. The `run_policy()` method handles the full loop:

```
observe → policy(obs) → send_action(action) → step physics → repeat
```

We'll also cover **domain randomization** (making the sim varied enough for
real-world transfer) and **dataset recording** (saving trajectories for training).

In [ ]:
from IPython.display import display, Image as IPImage

try:
    from strands_robots.simulation import create_simulation
    sim = create_simulation()
    sim.create_world(timestep=0.002)
    sim.add_robot("so100", data_config="so100", position=[0, 0, 0])
    sim.add_object("table", shape="box", position=[0.3, 0, -0.02],
                   size=[0.3, 0.3, 0.02], color=[0.6, 0.4, 0.2, 1], is_static=True)
    sim.add_object("red_cube", shape="box", position=[0.3, 0, 0.05],
                   size=[0.03, 0.03, 0.03], color=[1, 0, 0, 1])
    sim.step(n_steps=100)
    _mujoco_ok = True
except ImportError as e:
    print(f"⏭ MuJoCo not installed: {e}")
    print("  Install with: pip install 'strands-robots[sim-mujoco]'")
    _mujoco_ok = False

def show(sim, label=""):
    result = sim.render(camera_name="default", width=640, height=480)
    if result.get("status") == "success":
        for item in result.get("content", []):
            if "image" in item:
                display(IPImage(data=item["image"]["source"]["bytes"]))
                if label: print(label)
                return
    print("Rendering unavailable")

if _mujoco_ok:
    show(sim, "Initial scene")

## Running a Policy Loop

`run_policy()` orchestrates the full loop. Here we use the mock policy,
but it works the same with GR00T or any HuggingFace model:

In [ ]:
_mujoco_ok = globals().get('_mujoco_ok', False)
if _mujoco_ok:
    result = sim.run_policy(
        robot_name="so100",
        policy_provider="mock",            # swap to "groot" or a HF model name
        instruction="pick up the red cube",
        duration=2.0,                      # seconds of simulation time
    )
    print(result["content"][0]["text"])
    
    show(sim, "After 2s of mock policy execution")
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Domain Randomization

Real-world transfer requires training on varied environments. Randomization
changes colors, lighting, physics parameters, and object positions:

In [ ]:
_mujoco_ok = globals().get('_mujoco_ok', False)
if _mujoco_ok:
    result = sim.randomize(
        randomize_colors=True,       # random object/robot colors
        randomize_lighting=True,     # random light direction/intensity
        randomize_physics=True,      # friction, damping, mass variations
        randomize_positions=True,    # nudge objects ±noise
        position_noise=0.02,         # 2cm position noise
        seed=42,                     # reproducible for debugging
    )
    print(result["content"][0]["text"])
    
    show(sim, "After domain randomization — notice the color changes")
else:
    print('⏭ Skipped (MuJoCo not installed)')

## Recording Trajectories as LeRobot Datasets

Train new models from simulation data:

```python
# Start recording
sim.start_recording(
    repo_id="your-username/sim-pick-cube",
    task="pick up the red cube",
    fps=30,
    vcodec="libsvtav1",  # efficient video codec
)

# Run several episodes
for ep in range(100):
    sim.reset()
    sim.randomize(seed=ep)
    sim.run_policy("so100", "mock", instruction="pick up the red cube", duration=5.0)

# Stop and optionally push to HuggingFace
sim.stop_recording()
# sim.stop_recording(push_to_hub=True)
```

## Evaluation Pattern

A standard eval loop: run N episodes, check success, report rate:

```python
success_count = 0

for ep in range(50):
    sim.reset()
    sim.randomize(seed=ep)

    sim.run_policy(
        "so100",
        "lerobot/act_aloha_sim_transfer_cube_human",  # real policy
        instruction="pick up the red cube",
        duration=5.0,
    )

    # Check task success (e.g., cube height > threshold)
    obs = sim.get_observation("so100")
    # if cube_is_lifted(obs): success_count += 1

print(f"Success rate: {success_count}/50 = {success_count/50:.0%}")
```

In [ ]:
_mujoco_ok = globals().get('_mujoco_ok', False)
if _mujoco_ok:
    sim.destroy()
    print("Done ✅")
else:
    print('⏭ Skipped (MuJoCo not installed)')